In [1]:
import sys
sys.path.append("..")

%load_ext autoreload
%autoreload 2


from src.data import load_fd001, add_rul, DROP_COLS, FEATURES
from src.validation import run_cv

In [2]:
import numpy as np
import pandas as pd
train, test, rul = load_fd001()
train = add_rul(train)
print(train.shape)
print(train.groupby("unit")["RUL"].min().unique())
print(train["RUL"].max())

(20631, 27)
[0]
125


GroupKFold:

In [3]:
from sklearn.model_selection import GroupKFold
X = train[FEATURES]
y = train["RUL"]
groups = train["unit"]
gfk = GroupKFold(n_splits = 10)

In [4]:
splits = list(gfk.split(X,y, groups = groups))
print(len(splits))
print(splits[0])

10
(array([    0,     1,     2, ..., 20628, 20629, 20630], shape=(18559,)), array([ 4565,  4566,  4567, ..., 19008, 19009, 19010], shape=(2072,)))


In [5]:
for fold, (train_idx, val_idx) in enumerate(splits):
    train_units = set(groups.iloc[train_idx])
    val_units = set(groups.iloc[val_idx])
    overlap = train_units & val_units
    print(f"Fold {fold}: {len(val_units)} Triebwerke, {len(val_idx)} Zeilen, Überschneidung {len(overlap)}")

Fold 0: 10 Triebwerke, 2072 Zeilen, Überschneidung 0
Fold 1: 10 Triebwerke, 2057 Zeilen, Überschneidung 0
Fold 2: 10 Triebwerke, 2069 Zeilen, Überschneidung 0
Fold 3: 10 Triebwerke, 2066 Zeilen, Überschneidung 0
Fold 4: 10 Triebwerke, 2066 Zeilen, Überschneidung 0
Fold 5: 10 Triebwerke, 2055 Zeilen, Überschneidung 0
Fold 6: 10 Triebwerke, 2048 Zeilen, Überschneidung 0
Fold 7: 10 Triebwerke, 2056 Zeilen, Überschneidung 0
Fold 8: 10 Triebwerke, 2071 Zeilen, Überschneidung 0
Fold 9: 10 Triebwerke, 2071 Zeilen, Überschneidung 0


Formel für NASA-Score: 
$$s = \begin{cases} e^{-d/13} - 1 & \text{wenn } d < 0 \quad \text{(zu früh)} \\ e^{d/10} - 1 & \text{wenn } d \geq 0 \quad \text{(zu spät)} \end{cases}$$
mit d = Vorhersage - wahre RUL

In [6]:
print(f"10 Zyklen zu früh: {np.exp(10/13)-1:.2f} 30 Zyklen zu früh: {np.exp(30/13)-1:.2f}")
print(f"10 Zyklen zu spät: {np.exp(10/10)-1:.2f} 30 Zyklen zu spät: {np.exp(30/10)-1:.2f}")

10 Zyklen zu früh: 1.16 30 Zyklen zu früh: 9.05
10 Zyklen zu spät: 1.72 30 Zyklen zu spät: 19.09


In [7]:
from src.metrics import nasa_score

nasa_score([50],[40])

np.float64(1.1581055339484458)

In [8]:
nasa_score([50, 50], [40, 60])

np.float64(2.876387362407491)

In [9]:
nasa_score([50,50], [40,60], "mean")

np.float64(1.4381936812037455)

In [10]:
nasa_score([50,50],[50,50])

np.float64(0.0)

In [11]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

ergebnisse = []

for fold, (train_idx, val_idx) in enumerate(splits):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = LinearRegression()
    model.fit(X_tr,y_tr)
    pred_tr = model.predict(X_tr)
    pred_val = model.predict(X_val)
    d = pred_val - y_val

    ergebnisse.append({

        "fold": fold,
        "rmse_train": root_mean_squared_error(y_tr, pred_tr),
        "rmse_val": root_mean_squared_error(y_val, pred_val),
        "nasa_val": nasa_score(y_val, pred_val, "mean"),
        "bias": d.mean(),
        "anteil_spaet": (d>0).mean(),
        "max_spaet": d.max(),
    })

cv_lr = pd.DataFrame(ergebnisse)
cv_lr.drop(columns="fold").agg(["mean", "std"])

,rmse_train,rmse_val,nasa_val,bias,anteil_spaet,max_spaet
mean,21.485017,21.560367,9.509827,0.000667,0.523391,53.144905
std,0.149752,1.333467,3.260342,3.562756,0.067326,7.055259


In [12]:
from xgboost import XGBRegressor
from sklearn.metrics import root_mean_squared_error

ergebnisse = []

for fold, (train_idx, val_idx) in enumerate(splits):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = XGBRegressor()
    model.fit(X_tr,y_tr)
    pred_tr = model.predict(X_tr)
    pred_val = model.predict(X_val)
    d= pred_val - y_val

    ergebnisse.append({

        "fold": fold,
        "rmse_train": root_mean_squared_error(y_tr, pred_tr),
        "rmse_val": root_mean_squared_error(y_val, pred_val),
        "nasa_val": nasa_score(y_val, pred_val, "mean"),
        "bias": d.mean(),
        "anteil_spaet": (d>0).mean(),
        "max_spaet": d.max(),
    })

cv_xgb = pd.DataFrame(ergebnisse)
cv_xgb.drop(columns="fold").agg(["mean", "std"])

,rmse_train,rmse_val,nasa_val,bias,anteil_spaet,max_spaet
mean,11.950677,19.225841,18.774737,-0.096558,0.424170,76.982822
std,0.188642,2.442047,13.600800,2.923220,0.043026,11.040047


In [13]:
cv_xgb["nasa_val"]

0    46.607694
1    31.552723
2     6.392680
3    20.903616
4    11.162466
5     8.728299
6    16.592899
7     7.754675
8    30.939004
9     7.113314
Name: nasa_val, dtype: float64

In [14]:
vergleich = pd.DataFrame({
    "lr" :cv_lr["nasa_val"],
    "xgb": cv_xgb["nasa_val"],
})
vergleich

,lr,xgb
0,15.665354,46.607694
1,13.028939,31.552723
2,5.768951,6.392680
3,10.629681,20.903616
4,7.846484,11.162466
5,6.803896,8.728299
6,10.044791,16.592899
7,6.395592,7.754675
8,11.700797,30.939004
9,7.213782,7.113314


## Ergebnis: Baseline im GroupKFold (10 Folds nach Triebwerk)

| Modell | Train-RMSE | Val-RMSE | Lücke | NASA val (Mittel pro Vorhersage) |
|---|---|---|---|---|
| Lineare Regression | 21,49 ± 0,14 | 21,56 ± 1,27 | 0,08 | **9,49 ± 2,18** |
| XGBoost (Standardparameter) | 11,96 ± 0,19 | **19,32 ± 2,10** | 7,36 | 18,73 ± 8,15 |

*Mittelwert ± Standardabweichung über die 10 Folds.*

**Interpretation**
- Die lineare Regression underfittet (Train ≈ Val), XGBoost overfittet (Lücke 7,4 RMSE).
- Nach RMSE ist XGBoost rund 10 % besser, nach NASA-Score ist die lineare Regression etwa doppelt so gut. Welches Modell besser ist, hängt also von der Metrik ab.
- Die Ursache liegt nicht in der Richtung der Fehler: Beide Modelle sind im Mittel praktisch unverzerrt (Bias ≈ 0 bzw. −0,2 Zyklen). Entscheidend sind die Ausreißer. XGBoosts größter Verspätungsfehler liegt rund 30 Zyklen über dem der linearen Regression, und der NASA-Score bestraft große zu späte Vorhersagen exponentiell.
- Die Fold-Rangfolge ist bei beiden Modellen fast gleich (leichteste Folds 2, 5, 9, schwerste 0, 3). Auf leichten Folds liegt XGBoost gleichauf, auf schweren ist es etwa 2,5-mal so schlecht. XGBoost verstärkt also die Schwierigkeit ungewöhnlicher Triebwerke.
- Konsequenz für das Tuning: Ziel ist nicht nur ein niedrigerer RMSE, sondern vor allem weniger große Verspätungsfehler.

In [15]:
nasa_score([50,50],[40,60], "mean")

np.float64(1.4381936812037455)

In [16]:
nasa_score([50,50],[50,50])

np.float64(0.0)

In [17]:
cv_lr = run_cv(LinearRegression(), X, y, splits)
cv_lr.drop(columns="fold").agg(["mean", "std"])

,rmse_train,rmse_val,nasa_val,bias,anteil_spaet,max_spaet
mean,21.485017,21.560367,9.509827,0.000667,0.523391,53.144905
std,0.149752,1.333467,3.260342,3.562756,0.067326,7.055259


In [18]:
cv_xgb = run_cv(XGBRegressor(),X, y, splits)
cv_xgb.drop(columns="fold").agg(["mean", "std"])

,rmse_train,rmse_val,nasa_val,bias,anteil_spaet,max_spaet
mean,11.950677,19.225841,18.774737,-0.096558,0.424170,76.982822
std,0.188642,2.442047,13.600800,2.923220,0.043026,11.040047
